# My FindingZ analysis
**Question:** What are you trying to learn?

Use the Python environment containing FindingZ (on CIT: Python (hep)). Run cells from the top. FindingZ fills in your exported choices; edit them and rerun the cells below. A blank template waits for you to choose samples.

The only FindingZ helpers used here locate samples and load saved settings. The selections, weights, plots and statistical calculation are written below.

In [ ]:
import math
import numpy as np
import pandas as pd
from matplotlib.figure import Figure
from findingz.notebook_analysis import available_samples
from findingz.analysis_variables import load_variable_catalog

# Render Matplotlib figures inside Jupyter.
if "get_ipython" in globals():
    get_ipython().run_line_magic("matplotlib", "inline")

In [ ]:
analysis = {"plot": None, "count": None}
saved_plot = {}
saved_count = {}

## 1. Available samples
Copy sample IDs from the table. `sample.load()` reads the saved event table and supplies standard derived lepton columns; it applies no analysis cuts or luminosity scaling. Each row is an event with the saved dilepton candidate, not an individual particle.

In [ ]:
library = available_samples()
display(pd.DataFrame([
    {"ID": key, "name": sample.label, "cross section [pb]": sample.cross_section_pb,
     "generated events": sample.generated_events, "file": str(sample.path)}
    for key, sample in library.items()
]))
# Variable IDs may map to differently named columns in course configuration.
current_variables = {key: value.model_dump() for key, value in load_variable_catalog().variables.items()}
plot_variables = saved_plot.get("variables") or current_variables
count_variables = saved_count.get("variables") or current_variables

## 2. Select events and plot
`mll` is the pair invariant mass. Other basic choices are `leading_lepton_pt`, `subleading_lepton_pt`, `leading_lepton_eta`, and `subleading_lepton_eta`. Leading means higher pT.

Cuts are inclusive ranges; mass and pT are in GeV, eta is dimensionless. `{}` means no extra cuts. `channels = ["ee"]` selects electrons, `["mumu"]` muons; `None` keeps all and `[]` keeps none.

Luminosity is always in **fb⁻¹** here: 20 nb⁻¹ = 0.02 pb⁻¹ = `0.00002` fb⁻¹. It changes experimental exposure, not the number of generated Monte Carlo events. Shape plots normalize selected weights to sum to one.

In [ ]:
sample_ids = []
observable = "mll"
shape_only = True
luminosity_fb = 1.0

In [ ]:
cuts = {}  # Example: {"mll": (80, 100)}
channels = None

In [ ]:
frames = {}
plot_weights = {}
for sample_id in sample_ids:
    sample = library[sample_id]
    events = sample.load()
    mask = pd.Series(True, index=events.index)
    if channels is not None:
        mask &= events["channel"].isin(channels)
    for variable, (low, high) in cuts.items():
        column = plot_variables[variable]["column"]
        mask &= events[column].between(low, high, inclusive="both")
    selected = events.loc[mask].copy()
    weights = pd.to_numeric(selected.get("weight", pd.Series(1.0, index=selected.index)))
    assert np.isfinite(weights).all() and (weights >= 0).all(), "Expected finite non-negative event weights"
    if shape_only:
        if weights.sum() > 0:
            weights = weights / weights.sum()
    else:
        assert luminosity_fb >= 0
        assert sample.generated_events and sample.cross_section_pb is not None, "Missing normalization metadata"
        # pb × fb⁻¹ × 1000 gives events. Divide by ALL generated events,
        # not just the rows passing these cuts.
        weights = weights * sample.cross_section_pb * luminosity_fb * 1000 / sample.generated_events
    frames[sample_id] = selected
    plot_weights[sample_id] = weights

In [ ]:
if frames:
    column = plot_variables[observable]["column"]
    values = pd.to_numeric(pd.concat([frame[column] for frame in frames.values()]), errors="coerce")
    values = values[np.isfinite(values)]
    low, high = (float(values.min()), float(values.max())) if len(values) else (0.0, 1.0)
    if low == high:
        padding = max(abs(low) * 0.05, 0.5)
        low, high = low - padding, high + padding
    bins = np.linspace(low, high, 41)
    figure = Figure(figsize=(7, 4))
    ax = figure.subplots()
    for sample_id, selected in frames.items():
        ax.hist(selected[column], bins=bins, weights=plot_weights[sample_id],
                histtype="step", linewidth=2, label=library[sample_id].label)
    ax.set_xlabel(plot_variables[observable]["label"])
    ax.set_ylabel("Fraction of selected sample" if shape_only else "Expected events")
    ax.legend()
    figure.tight_layout()
    display(figure)
else:
    print("Choose sample IDs above to make a plot.")

## 3. Count events under two hypotheses
For a Z search, choose QED-only as the **null** and the complete Standard Model prediction as the **alternative**. Never add them: the SM already includes photon and Z exchange and interference.

Use the same final state, beam energy, detector and generation acceptance. Comparing two random samples of the same physics is not a discovery test.

Counting choices below are independent of plotting. To copy them, set `count_cuts = dict(cuts)`, `count_channels = channels`, and `count_luminosity_fb = luminosity_fb`.

In [ ]:
null_id = None
alternative_id = None
count_cuts = {}
count_channels = None
count_luminosity_fb = 1.0
null_uncertainty = 0.0

In [ ]:
counts = {}
if null_id is not None and alternative_id is not None:
    assert null_id != alternative_id, "Choose two different samples"
    assert library[null_id].analysis_context == library[alternative_id].analysis_context, "Configurations differ"
    assert count_luminosity_fb >= 0
    for role, sample_id in [("null", null_id), ("alternative", alternative_id)]:
        sample = library[sample_id]
        assert sample.generated_events and sample.cross_section_pb is not None, "Missing normalization metadata"
        events = sample.load()
        mask = pd.Series(True, index=events.index)
        if count_channels is not None:
            mask &= events["channel"].isin(count_channels)
        for variable, (low, high) in count_cuts.items():
            column = count_variables[variable]["column"]
            mask &= events[column].between(low, high, inclusive="both")
        selected = events.loc[mask]
        weights = pd.to_numeric(selected.get("weight", pd.Series(1.0, index=selected.index)))
        assert np.isfinite(weights).all() and (weights >= 0).all(), "Expected finite non-negative weights"
        counts[role] = weights.sum() * sample.cross_section_pb * count_luminosity_fb * 1000 / sample.generated_events
        print(f"{role}: {len(selected)} selected simulation rows; {counts[role]:.2f} expected events")
else:
    print("Choose null_id and alternative_id above to compare predictions.")

### Expected sensitivity, not observed discovery
We pretend the count equals the alternative prediction, without random fluctuations: the **Asimov approximation**. For a known null rate, the signed Poisson sensitivity is

$$Z_A = \mathrm{sgn}(N_1-N_0)\sqrt{2[N_1\ln(N_1/N_0)-N_1+N_0]}.$$

Start with `null_uncertainty = 0.0`. The optional branch below fits the null rate with a Gaussian constraint (e.g. `0.10` means 10%). It keeps exported web analyses reproducible; it is not finite-Monte-Carlo uncertainty.

A deficit is not an excess discovery. With small counts this sigma is approximate, not an exact p-value; observed data require integer counts. Zero simulated background does not prove a zero physical background. Monte Carlo statistical uncertainty is omitted.

In [ ]:
if counts:
    n0, n1 = counts["null"], counts["alternative"]
    difference = n1 - n0
    assert 0 <= null_uncertainty <= 1
    z_expected = None
    if n0 > 0:
        fitted_null = n0
        penalty = 0.0
        if null_uncertainty > 0:
            variance = (null_uncertainty * n0) ** 2
            # Maximize Poisson(n1 | b) × Gaussian(b | n0, variance).
            offset = n0 - variance
            root = math.hypot(offset, 2 * math.sqrt(n1 * variance))
            fitted_null = ((offset + root) / 2 if offset >= 0
                           else 2 * n1 * variance / (root - offset))
            penalty = (fitted_null - n0) ** 2 / variance
        # For n1 = 0, the limiting value of n1*log(n1/b) is zero.
        q = (2 * fitted_null if n1 == 0 else
             2 * (n1 * math.log1p((n1 - fitted_null) / fitted_null) - n1 + fitted_null))
        z_expected = math.copysign(math.sqrt(max(0.0, q + penalty)), difference)
    print(f"Alternative minus null: {difference:.2f} expected events")
    if z_expected is None:
        print("Sensitivity undefined: zero predicted null count.")
    else:
        print(f"Expected signed sensitivity: {z_expected:+.2f} sigma (not observed significance)")
        if min(n0, n1) < 10:
            print("Small counts: the Asimov sigma is only an approximation.")

## 4. Your conclusion
What did your cuts change? Which assumptions limit the result?

Read [PDG Statistics](https://pdg.lbl.gov/2025/reviews/rpp2025-rev-statistics.pdf), sections 40.1 and 40.3, for likelihoods and p-values; [Cowan et al.](https://arxiv.org/abs/1007.1727) for the Asimov calculation. A p-value is not the probability that the null hypothesis is true.

Save the notebook with its outputs. Submit its matching `.settings.json` file too, if exported from FindingZ. Sample event files must remain available; they are not embedded.